In [92]:
# https://platform.olimpiada-ai.ro/ro/problems/184

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
from tqdm.auto import tqdm

In [93]:
train = pd.read_csv("/kaggle/input/datasets/abukanabek/galactic-wars/train_galactic_wars.csv")
test = pd.read_csv("/kaggle/input/datasets/abukanabek/galactic-wars/test_galactic_wars.csv")

train['surprise_attack'] = train['surprise_attack'].astype(int)
test['surprise_attack'] = test['surprise_attack'].astype(int)

train.shape, test.shape

((4999, 15), (4001, 14))

In [94]:
train.head()

,FightID,weapon_jedai,armour_jedai,weapon_calmtrooper,armour_calmtrooper,injuries,number_of_fights,force_level,accuracy_calmtrooper,fight_planet,weather_conditions,surprise_attack,moral_jedai,moral_calmtrooper,winner
0,640,Single Lightsaber,48.7%,Standard Blaster,79.2%,Jedai,151.4,28.1,75.0,Kamira,Storm,0,26.8,73.4,0
1,6876,Ancient Jedai Blade,73.8%,Heavy Blaster,66.9%,Both,67.4,58.0,59.1,Corsavant,Storm,1,53.9,64.7,0
2,7664,Ancient Jedai Blade,60.9%,Heavy Blaster,66.2%,Both,3.0,53.3,62.9,Tavooine,Sandstorm,0,51.0,57.5,0
3,4463,Double Lightsaber,80.3%,Experimental Weapon,37.2%,Calmtrooper,45.8,81.7,47.4,Kamira,Sandstorm,1,69.4,37.8,1
4,461,Single Lightsaber,54.9%,Standard Blaster,83.2%,Jedai,71.5,39.3,71.6,Kamira,Snow,1,52.4,77.1,0


In [95]:
train['winner'].value_counts(normalize=True)

winner
0    0.558312
1    0.441688
Name: proportion, dtype: float64

In [96]:
pd.concat([train['weapon_calmtrooper'].value_counts(), test['weapon_calmtrooper'].value_counts()], axis=1)

,count,count
weapon_calmtrooper,,
Standard Blaster,1716.0,1284.0
Experimental Weapon,1646.0,NaN
Heavy Blaster,1637.0,1363.0
Double Blaster,NaN,1354.0


In [97]:
pd.concat([train['weapon_jedai'].value_counts(), test['weapon_jedai'].value_counts()], axis=1)

,count,count
weapon_jedai,,
Single Lightsaber,1716,1284
Double Lightsaber,1646,1354
Ancient Jedai Blade,1637,1363


In [98]:
weapon_ans = (train['weapon_calmtrooper'] == 'Experimental Weapon').sum().item()
train['weapon_calmtrooper'] = train['weapon_calmtrooper'].map(lambda x: x if x != 'Experimental Weapon' else 'Double Blaster')
weapon_ans

1646

In [99]:
nabira_ans = ((test['fight_planet']=='Nabira') & (test['weather_conditions']=='Snow')).sum().item()
nabira_ans

143

In [100]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

clusters = le.fit_transform(test['weapon_jedai'])

In [101]:
from sklearn.model_selection import train_test_split

id_col = 'FightID'
target_col = 'winner'
features = [c for c in train.columns if c not in [id_col, target_col]]
cat_features = [c for c in train.select_dtypes('object').columns if c in features]

X, y = train[features], train[target_col]

X_train, X_valid, y_train, y_valid = train_test_split(X, y, stratify=y, test_size=0.1, random_state=42)
X_train.shape, X_valid.shape

((4499, 13), (500, 13))

In [102]:
from catboost import Pool

train_pool = Pool(X_train, y_train, cat_features=cat_features)
valid_pool = Pool(X_valid, y_valid, cat_features=cat_features)

In [103]:
from catboost import CatBoostClassifier

params = {
    'iterations': 1000,
    'loss_function': 'Logloss',
    'eval_metric': 'TotalF1:average=Macro',
    'metric_period': 100,
    'max_depth': 2,
    'random_state': 42
}

model = CatBoostClassifier(**params)

model.fit(train_pool, eval_set=valid_pool)

Learning rate set to 0.045944
0:	learn: 0.8790680	test: 0.8929107	best: 0.8929107 (0)	total: 5.56ms	remaining: 5.55s
100:	learn: 0.9708790	test: 0.9694843	best: 0.9694843 (100)	total: 461ms	remaining: 4.1s
200:	learn: 0.9754156	test: 0.9735806	best: 0.9735806 (200)	total: 925ms	remaining: 3.68s
300:	learn: 0.9792487	test: 0.9695160	best: 0.9735806 (200)	total: 1.38s	remaining: 3.2s
400:	learn: 0.9839949	test: 0.9756252	best: 0.9756252 (400)	total: 1.82s	remaining: 2.71s
500:	learn: 0.9855721	test: 0.9735806	best: 0.9756252 (400)	total: 2.26s	remaining: 2.25s
600:	learn: 0.9866972	test: 0.9756002	best: 0.9756252 (400)	total: 2.71s	remaining: 1.8s
700:	learn: 0.9873743	test: 0.9796669	best: 0.9796669 (700)	total: 3.16s	remaining: 1.35s
800:	learn: 0.9891779	test: 0.9817096	best: 0.9817096 (800)	total: 3.63s	remaining: 903ms
900:	learn: 0.9905317	test: 0.9796669	best: 0.9817096 (800)	total: 4.12s	remaining: 453ms
999:	learn: 0.9916593	test: 0.9776218	best: 0.9817096 (800)	total: 4.57s	rem

In [104]:
y_pred = model.predict(test[features])

subm = pd.DataFrame({
    'subtaskID': [1, 2] + [3] * len(test) + [4] * len(test),
    'datapointID': [1, 2] + test[id_col].tolist() + test[id_col].tolist(),
    'answer': [nabira_ans, weapon_ans] + clusters.tolist() + y_pred.tolist()
})

subm.to_csv("submission.csv", index=False)
subm

,subtaskID,datapointID,answer
0,1,1,143
1,2,2,1646
2,3,1596,2
3,3,4432,1
4,3,486,2
...,...,...,...
7999,4,4843,1
8000,4,6117,1
8001,4,5810,1
8002,4,3589,1
